# 순위 가중치 완화 실험

- 실험명: weight_difference
- 순위 가중치: 1순위 1.5, 2순위 1.25, 3순위 1.0
- 데이터 분리: train/test = 8:2, train 내부 train/valid = 8:2
- 검증: shuffle + stratify split, 3-Fold Stratified K-Fold


## 1. 패키지 및 경로

- 별도 CSV 산출물은 저장하지 않음.
- 실험 결과는 노트북 출력으로만 확인함.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

BASE_PATH = Path.cwd()
while BASE_PATH.name != "oracle_mnc_project" and BASE_PATH.parent != BASE_PATH:
    BASE_PATH = BASE_PATH.parent

RAW_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "source" / "leisure_activity_survey_2021_2025_selected_columns_enriched.csv"
MAPPING_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "processed" / "satisfaction" / "ml_activity_category_mapping.csv"

print("RAW_PATH 존재:", RAW_PATH.exists())
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())


## 2. 선호도 순위 테이블 생성

- 향후 희망 여가활동 1~3순위를 사용함.
- 음악, 체육용품, 여행사, 교통수단, 분류범위외는 제외함.
- 같은 응답자 안에서 동일 중분류가 반복되면 1회만 반영함.


In [ ]:
raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")
mapping.columns = ["activity_code", "activity_name", "category", "use_target"]

excluded_categories = ["분류범위외", "교통수단", "여행사", "음악", "체육용품"]
mapping["use_target_final"] = (
    mapping["use_target"].astype(bool)
    & ~mapping["category"].isin(excluded_categories)
)

rank_cols = {
    1: "향후 희망하는 여가활동 1순위",
    2: "향후 희망하는 여가활동 2순위",
    3: "향후 희망하는 여가활동 3순위",
}
rank_score = {1: 1.5, 2: 1.25, 3: 1.0}

preference_raw = raw.loc[
    raw["조사년도"].isin([2024, 2025]),
    ["응답자_ID", "최종가중치", "성별", "연령", "조사년도"] + list(rank_cols.values())
].copy()
preference_raw["성별_연령"] = (
    preference_raw["성별"].astype("Int64").astype(str)
    + "_"
    + preference_raw["연령"].astype("Int64").astype(str)
)

code_to_category = mapping.set_index("activity_code")["category"].to_dict()
code_to_use = mapping.set_index("activity_code")["use_target_final"].to_dict()

wide_records = []
long_records = []

for _, row in preference_raw.iterrows():
    valid_rows = []
    seen_categories = set()
    excluded_count = 0
    duplicate_count = 0

    for rank_no, col in rank_cols.items():
        activity_code = row[col]
        if pd.isna(activity_code):
            continue

        activity_code = int(activity_code)
        category = code_to_category.get(activity_code)
        use_target = bool(code_to_use.get(activity_code, False))

        if not use_target:
            excluded_count += 1
            continue

        if category in seen_categories:
            duplicate_count += 1
            continue

        seen_categories.add(category)
        valid_rows.append({
            "rank_no": rank_no,
            "rank_score": rank_score[rank_no],
            "target_category": category,
        })

    score_sum = sum(x["rank_score"] for x in valid_rows)

    wide_row = {
        "응답자_ID": row["응답자_ID"],
        "성별": row["성별"],
        "연령": row["연령"],
        "조사년도": row["조사년도"],
        "성별_연령": row["성별_연령"],
        "최종가중치": row["최종가중치"],
        "선호_유효순위수": len(valid_rows),
        "제외분류_제거수": excluded_count,
        "중복중분류_제거수": duplicate_count,
    }

    for i, valid in enumerate(valid_rows, start=1):
        wide_row[f"선호_유효중분류_{i}순위"] = valid["target_category"]
        long_records.append({
            "응답자_ID": row["응답자_ID"],
            "성별": row["성별"],
            "연령": row["연령"],
            "조사년도": row["조사년도"],
            "성별_연령": row["성별_연령"],
            "최종가중치": row["최종가중치"],
            "rank_no": valid["rank_no"],
            "rank_score": valid["rank_score"],
            "target_category": valid["target_category"],
            "rank_weight_share": valid["rank_score"] / score_sum if score_sum > 0 else 0,
            "sample_weight": row["최종가중치"] * valid["rank_score"] / score_sum if score_sum > 0 else 0,
        })

    wide_records.append(wide_row)

rank_base = pd.DataFrame(wide_records)
rank_long = pd.DataFrame(long_records)

print("응답자 테이블:", rank_base.shape)
print("순위 long 테이블:", rank_long.shape)
print("\n유효순위수 분포")
print(rank_base["선호_유효순위수"].value_counts().sort_index())

weighted_dist = (
    rank_long.groupby("target_category", as_index=False)["sample_weight"].sum()
    .rename(columns={"sample_weight": "weighted_n"})
)
weighted_dist["share"] = weighted_dist["weighted_n"] / weighted_dist["weighted_n"].sum()
display(weighted_dist.sort_values("share", ascending=False))


## 3. 학습/검증/테스트 분리

- 전체 유효 응답자 기준 train_valid 80%, test 20%로 분리함.
- train_valid 안에서 train 80%, valid 20%로 다시 분리함.
- 조사년도와 1순위 유효중분류를 함께 stratify 기준으로 사용함.


In [ ]:
primary_target = (
    rank_long.sort_values(["응답자_ID", "rank_no"])
    .groupby("응답자_ID", as_index=False)["target_category"]
    .first()
    .rename(columns={"target_category": "primary_target"})
)

model_base = (
    rank_base.loc[rank_base["선호_유효순위수"] > 0]
    .merge(primary_target, on="응답자_ID", how="left")
    .copy()
)

def choose_strata(df, min_count=2):
    year_target = df["조사년도"].astype(str) + "_" + df["primary_target"].astype(str)
    if year_target.value_counts().min() >= min_count:
        return year_target

    target_only = df["primary_target"].astype(str)
    if target_only.value_counts().min() >= min_count:
        return target_only

    return None

train_valid_base, test_base = train_test_split(
    model_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(model_base, 2),
)

train_base, valid_base = train_test_split(
    train_valid_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(train_valid_base, 2),
)

train_data = rank_long[rank_long["응답자_ID"].isin(train_base["응답자_ID"])].copy()
valid_data = rank_long[rank_long["응답자_ID"].isin(valid_base["응답자_ID"])].copy()
test_data = rank_long[rank_long["응답자_ID"].isin(test_base["응답자_ID"])].copy()

classes = np.array(sorted(rank_long["target_category"].unique()))
model_feature_cols = ["성별", "연령", "조사년도", "성별_연령"]

split_summary = pd.DataFrame({
    "dataset": ["train", "valid", "test"],
    "respondents": [
        train_base["응답자_ID"].nunique(),
        valid_base["응답자_ID"].nunique(),
        test_base["응답자_ID"].nunique(),
    ],
})
split_summary["share"] = split_summary["respondents"] / model_base["응답자_ID"].nunique()

print("학습 레코드:", train_data.shape)
print("검증 레코드:", valid_data.shape)
print("테스트 레코드:", test_data.shape)
display(split_summary)

print("\n연도별 분포")
display(
    pd.concat([
        train_base.assign(dataset="train"),
        valid_base.assign(dataset="valid"),
        test_base.assign(dataset="test"),
    ]).pivot_table(index="dataset", columns="조사년도", values="응답자_ID", aggfunc="count", fill_value=0)
)


## 4. 모델 정의 및 평가 함수

- Prior, Multinomial Logistic, Balanced Logistic, RandomForest light, ExtraTrees light, HistGradientBoosting light를 비교함.
- 현재 환경 실행시간을 고려해 트리 계열은 light 설정으로 실험함.


In [ ]:
class WeightedPriorModel:
    def fit(self, y, sample_weight):
        y = pd.Series(y)
        w = pd.Series(sample_weight)
        prior = w.groupby(y).sum() / w.sum()
        self.classes_ = np.array(sorted(y.unique()))
        self.prior_ = prior.reindex(self.classes_, fill_value=0).to_numpy()
        return self

    def predict_proba(self, X):
        return np.tile(self.prior_, (len(X), 1))

def make_models():
    onehot = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), model_feature_cols)],
        remainder="drop",
    )

    return {
        "weighted_prior": WeightedPriorModel(),
        "multinomial_logistic": Pipeline([
            ("preprocess", onehot),
            ("model", LogisticRegression(max_iter=1000, solver="lbfgs", C=1.0))
        ]),
        "balanced_logistic": Pipeline([
            ("preprocess", onehot),
            ("model", LogisticRegression(max_iter=1000, solver="lbfgs", C=1.0, class_weight="balanced"))
        ]),
        "random_forest_light": Pipeline([
            ("preprocess", onehot),
            ("model", RandomForestClassifier(
                n_estimators=20,
                max_depth=4,
                min_samples_leaf=100,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=1,
            ))
        ]),
        "extra_trees_light": Pipeline([
            ("preprocess", onehot),
            ("model", ExtraTreesClassifier(
                n_estimators=20,
                max_depth=4,
                min_samples_leaf=100,
                class_weight="balanced",
                random_state=42,
                n_jobs=1,
            ))
        ]),
        "hist_gradient_boosting_light": Pipeline([
            ("preprocess", onehot),
            ("model", HistGradientBoostingClassifier(
                max_iter=25,
                learning_rate=0.06,
                max_leaf_nodes=8,
                l2_regularization=1.0,
                random_state=42,
            ))
        ]),
    }

def fit_model(model, data):
    x = data[model_feature_cols].astype(str)
    y = data["target_category"]
    w = data["sample_weight"]

    if isinstance(model, WeightedPriorModel):
        return model.fit(y, w)

    return model.fit(x, y, model__sample_weight=w)

def align_proba(model, proba):
    model_classes = model.classes_ if isinstance(model, WeightedPriorModel) else model.named_steps["model"].classes_
    aligned = np.zeros((proba.shape[0], len(classes)))
    class_to_idx = {c: i for i, c in enumerate(model_classes)}

    for j, c in enumerate(classes):
        if c in class_to_idx:
            aligned[:, j] = proba[:, class_to_idx[c]]

    row_sum = aligned.sum(axis=1, keepdims=True)
    return np.divide(aligned, row_sum, out=np.zeros_like(aligned), where=row_sum > 0)

def weighted_brier(y_true, proba, sample_weight):
    y_index = pd.Categorical(y_true, categories=classes).codes
    y_onehot = np.zeros_like(proba)
    y_onehot[np.arange(len(y_index)), y_index] = 1
    return np.average(((proba - y_onehot) ** 2).sum(axis=1), weights=sample_weight)

def evaluate(model_name, model, data, dataset_name):
    x = data[model_feature_cols].astype(str)
    y = data["target_category"].to_numpy()
    w = data["sample_weight"].to_numpy()
    proba = align_proba(model, model.predict_proba(x))
    y_pred = classes[np.argmax(proba, axis=1)]

    return {
        "model": model_name,
        "dataset": dataset_name,
        "LogLoss": log_loss(y, proba, labels=classes, sample_weight=w),
        "Top1_Accuracy": accuracy_score(y, y_pred, sample_weight=w),
        "Top3_HitRate": top_k_accuracy_score(y, proba, k=3, labels=classes, sample_weight=w),
        "Macro_F1": f1_score(y, y_pred, labels=classes, average="macro", sample_weight=w, zero_division=0),
        "Weighted_F1": f1_score(y, y_pred, labels=classes, average="weighted", sample_weight=w, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y, y_pred, sample_weight=w),
        "Brier": weighted_brier(y, proba, w),
    }, y_pred

def category_metric(model_name, dataset_name, data, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        data["target_category"],
        y_pred,
        labels=classes,
        sample_weight=data["sample_weight"],
        zero_division=0,
    )

    return pd.DataFrame({
        "model": model_name,
        "dataset": dataset_name,
        "category": classes,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Actual_Weight": support,
        "Pred_Weight": [data.loc[y_pred == c, "sample_weight"].sum() for c in classes],
    })


## 5. 3-Fold Stratified K-Fold

- train_valid 80% 내부에서 3-Fold 검증을 수행함.


In [ ]:
cv_base = train_valid_base.reset_index(drop=True)
cv_strata = choose_strata(cv_base, min_count=3)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_rows = []

for fold_no, (tr_idx, va_idx) in enumerate(skf.split(cv_base, cv_strata), start=1):
    tr_ids = cv_base.iloc[tr_idx]["응답자_ID"]
    va_ids = cv_base.iloc[va_idx]["응답자_ID"]

    fold_train_data = rank_long[rank_long["응답자_ID"].isin(tr_ids)].copy()
    fold_valid_data = rank_long[rank_long["응답자_ID"].isin(va_ids)].copy()

    for model_name, model in make_models().items():
        print(f"fold {fold_no}: {model_name}")
        model = fit_model(model, fold_train_data)
        row, _ = evaluate(model_name, model, fold_valid_data, "cv_valid")
        row["fold"] = fold_no
        cv_rows.append(row)

cv_performance = pd.DataFrame(cv_rows)
cv_summary = cv_performance.groupby("model", as_index=False).agg(
    LogLoss_mean=("LogLoss", "mean"),
    LogLoss_std=("LogLoss", "std"),
    Top1_Accuracy_mean=("Top1_Accuracy", "mean"),
    Top3_HitRate_mean=("Top3_HitRate", "mean"),
    Macro_F1_mean=("Macro_F1", "mean"),
    Balanced_Accuracy_mean=("Balanced_Accuracy", "mean"),
    Brier_mean=("Brier", "mean"),
)

display(cv_summary.sort_values("LogLoss_mean"))


## 6. Holdout 성능 평가

- train 64%로 최종 모델을 학습함.
- valid 16%, test 20%에서 성능을 비교함.


In [ ]:
performance_rows = []
category_rows = []

for model_name, model in make_models().items():
    print("holdout:", model_name)
    model = fit_model(model, train_data)

    for dataset_name, dataset in [
        ("train", train_data),
        ("valid", valid_data),
        ("test", test_data),
    ]:
        row, y_pred = evaluate(model_name, model, dataset, dataset_name)
        performance_rows.append(row)
        category_rows.append(category_metric(model_name, dataset_name, dataset, y_pred))

performance = pd.DataFrame(performance_rows)
category_performance = pd.concat(category_rows, ignore_index=True)

baseline_logloss = performance.loc[
    (performance["model"] == "weighted_prior") & (performance["dataset"] == "test"),
    "LogLoss"
].iloc[0]
performance["LogLoss_Skill_vs_Prior"] = np.where(
    performance["dataset"] == "test",
    1 - performance["LogLoss"] / baseline_logloss,
    np.nan
)

test_summary = performance.loc[performance["dataset"] == "test"].sort_values("LogLoss")
display(test_summary)

best_model = test_summary.iloc[0]["model"]
print("best model:", best_model)
display(
    category_performance.loc[
        (category_performance["dataset"] == "test")
        & (category_performance["model"] == best_model)
    ].sort_values("F1", ascending=False)
)
